<img src='../OUTILS/bandeau_MF.png' align='right' width='100%'/>

# <div style='background-color: #27ae60; color: white; padding: 20px; border-radius: 10px; text-align: center;'>🌍🛰️ Manipulation de données satellitaires - Données Eumetsat - Format NetCDF</div>

---

## 🎯 **Objectifs du TP**

##   🔎 Analyser un fichier NetCDF type Chunk Meteosat-12 Eumetsat
##   🖼️ Produire une image à partir d'un canal 
##   🎚️ Modifier la dynamique des valeurs 
##   🌈 Appliquer des corrections gamma 
##   📍 Extraire une valeur ponctuelle 

---

## 📑 **Table des matières**

| Section | Titre |
|---------|-------|
| 🔗 | [**1.** 🔎 Visualisation du contenu du NetCDF](#1-visualisation-du-contenu) |
| 🔗 | [**2.** 🖼️ Production d'une image TIF (canal VIS006)](#2-production-dune-image-tif-canal-vis006) |
| 🔗 | [**2.1** 🎨 Étirement linéaire](#21-etirement-lineaire) |
| 🔗 | [**2.2** 📐 Correction Gamma](#22-correction-gamma) |
| 🔗 | [**3.** 🌡️ Production d'une image TIF (canal IR_105)](#3-production-dune-image-tif-canal-ir_105) |
| 🔗 | [**4.** 📍 Extraction de valeurs ponctuelles](#4-extraction-de-valeurs-ponctuelles) |
| 🔗 | [**5.** 📍 Extraction de valeurs ponctuelles à partir des TIF](#5-extraction-de-valeurs-ponctuelles) |
| 🔗 | [**6.** 📍 Exemples réalisés à partir d'extractions](#6-exemples) |

---

## 🧰 **Workflow : lignes de commandes bash (GDAL & ImageMagick)**

Ce TP utilise en grande partie la bibliothèque **GDAL** (*Geospatial Data Abstraction Library*), un outil puissant pour manipuler les données issues de satellites météorologiques.

Les fichiers utilisés sont au format **NetCDF**, issus de la production opérationnelle de Eumetsat.

### 📦 **Outils utilisés**

| Outil | Rôle | Commande typique |
|-------|------|------------------|
| **GDAL** | Manipulation de données géospatiales | `gdalinfo`, `gdal_translate` |
| **ImageMagick** | Traitement d'images | `convert`, `identify` |
| **NetCDF** | Format de données scientifiques | Lecture directe via GDAL |

---


<div class="alert alert-info" role="alert">
<h3> ⚙️ Initialisation de l'environnement</h3>
Importation des librairies nécessaires et configuration des chemins d'accès.
</div>

In [ ]:
# 📚 Import des bibliothèques
from datetime import datetime
import sys
import os
from osgeo import gdal
import subprocess
from IPython.display import display, HTML
import glob
from IPython.display import Image as IPImage
from PIL import Image as PILImage

# 🔧 Configuration des chemins d'accès GDAL
os.environ['PATH'] = f"/opt/conda/env_MF_teledetection/bin:{os.environ['PATH']}" 
os.environ['PATH'] = f"~/.conda/envs/env_MF_teledetection/bin:{os.environ['PATH']}"
os.environ['GDAL_DATA'] = '/opt/conda/env_MF_teledetection/share/gdal'
os.environ['PROJ_LIB'] = '/opt/conda/env_MF_teledetection/share/proj'

print("✅ Environnement configuré avec succès")

## 📡 **Rappel des différents canaux**

Le FCI (Flexible Combined Imager) de MTG dispose de 16 canaux spectraux :

<div align="center">
<a href="../DOCS/mtg_fci_synthese_produits.jpg" target="_blank">
    <img src='../DOCS/mtg_fci_synthese_produits.jpg' alt='Canaux MTG' width='900px'>
</a>
</div>

In [ ]:
cd ~/MF_DATA_MANIPULATION

In [ ]:
# 📁 Configuration des répertoires
#❗ Attention : ce chemin doit-être modifié pour toute arborescence différente
RepSource = "../stockage/DATA/20260614"
#RepSource = "/stockage/DATA/20260614"

# Création du dossier de résultats
!mkdir -p RESULTS
!mkdir -p RESULTS/tmp

output = 'RESULTS'

print("📁 Fichiers configurés :")
print(f"   - Chunks Meteosat-12 : {RepSource}")
print(f"   - Dossier de sortie : {output}")

<div id="1-visualisation-du-contenu"></div>
<div class="alert alert-info alert-success">
<h3 id="1"> 1️⃣ - 🔎 Visualisation du contenu du NetCDF</h3>
</div>

In [ ]:
# 📊 Nombre total de chunks disponibles
print("📊 Nombre de fichiers chunks disponibles :")
!ls -ltr {RepSource} | wc -l

In [ ]:
# 📋 Aperçu des premiers fichiers
print("📋 Aperçu des premiers chunks :")
!ls -ltr {RepSource} | head -n 5

### 📡 Analyse détaillée d'un chunk

Un chunk représente une bande de 300 lignes du disque complet (11136 colonnes).

La commande `gdalinfo` permet d'afficher les métadonnées d'un fichier NetCDF. Dans Jupyter, on la précède de `!` pour l'exécuter dans le shell.

In [ ]:
# 🔍 Analyse d'un chunk spécifique
chunk1 = f"{RepSource}/W_XX-EUMETSAT-Darmstadt,IMG+SAT,MTI1+FCI-1C-RRAD-FDHSI-FD--CHK-BODY---NC4E_C_EUMT_20260614120915_IDPFI_OPE_20260614120629_20260614120726_N__O_0073_0028.nc"

print("🔍 Métadonnées complètes du chunk :\n")
!gdalinfo {chunk1}

## 📡 Décryptage du fichier netCDF FCI (MTG)

| Information | Valeur | Signification |
|-------------|--------|----------------|
| 🏷️ **Format** | netCDF (NC4) | Données scientifiques multicouches, non géoréférencées directement |
| 🛰️ **Satellite / Instrument** | MTI1 (MTG) / FCI | Meteosat Third Generation, Flexible Combined Imager |
| 📊 **Niveau de produit** | 1C - RRAD | Radiances rectifiées, géolocalisées mais non projetées |
| 📐 **Dimensions réelles** | **300 × 11136** pixels par canal | 300 lignes de balayage, largeur du disque complet |
| 🌐 **Couverture géographique** | Lat : 18.48°–24.78°N<br>Lon : -79.64°–78.45°E | Vue partielle (haute latitude), pas le disque entier |
| 🕒 **Date / Heure** | 2026-06-14 12:00:00 UTC | Heure du slot (12h) |
| 🎯 **Projection** | GEOS (Géostationnaire) | Longitude nominale 0°, hauteur 35 785 834 m |
| 🌈 **Canaux disponibles** | 16 canaux (vis, nIR, IR, WV) | De 0.4 µm à 13.3 µm, dont le WV 6.3 & 7.3 µm |
| 🧩 **Sous-ensembles** | `/data/vis_04/measured/effective_radiance` | Chaque canal contient radiance, qualité, index, swath |
| 📏 **Type de données** | uint16 (radiances), uint8 (qualité) | Radiances en comptages numériques (non calibrées en °C) |


### 📋 Liste des canaux disponibles

La commande suivante extrait les noms des canaux FCI présents dans le fichier.

In [ ]:
# 🔍 Extraction des noms de canaux
print("📋 Canaux FCI disponibles dans le chunk :")
!gdalinfo {chunk1} 2>/dev/null | grep -E "FCI FDHSI" 

In [ ]:
# 🔍 Analyse détaillée du canal VIS006
print("🔍 Métadonnées du canal VIS006 :\n")
!gdalinfo -nomd NETCDF:{chunk1}:/data/vis_06/measured/effective_radiance

# Remarque : l'option -nomd est importante pour limiter le résultat aux seules informations du canal recherché

## 📡 Décryptage du canal VIS006 (FCI – MTG)

| Information | Valeur | Signification |
|-------------|--------|----------------|
| 🏷️ **Format** | netCDF4 | Données scientifiques hiérarchisées (groupe `/data/vis_06/measured/`) |
| 📐 **Dimensions** | **11136 × 300** pixels | Largeur du disque complet (colonnes) × nombre de lignes du chunk |
| 🛰️ **Projection** | GEOS (Sweep Y) | Vue géostationnaire, satellite MTG, longitude nominale 0° |
| 📏 **Pixel Size** | **0.0000279436 rad** | Résolution angulaire (radians) – très fine pour du visible |
| 📍 **Coordonnées** | x : -0.1556 à 0.1556 rad<br>y : 0.0556 à 0.0639 rad | Coordonnées projetées (en radians) correspondant aux angles de balayage |
| 📊 **Type de données** | **UInt16** (entier non signé 16 bits) | Comptages numériques bruts, à convertir en radiance |
| 📏 **Échelle & Offset** | scale = 0.00709761<br>offset = -1.4479115 | Formule : <br>**Radiance = count × scale + offset** |
| 🌈 **Grandeur physique** | `mW.m⁻².sr⁻¹.(cm⁻¹)⁻¹` | Radiance efficace (monochromatique) dans le visible |
| 🚫 **Valeur de remplissage** | 65535 | Pixels hors disque ou non valides |
| 🌐 **Couverture géographique** | Lat : 18.5°–24.8°N<br>Lon : -79.6°–78.5°E | Vue partielle (région Europe/Afrique/Atlantique) |
| 🕒 **Date / Heure** | 2026-06-14 12:00 UTC | Heure du slot (midi) |

In [ ]:
# 📊 Statistiques des canaux
print("📊 Statistiques des canaux :")
print("\n🔵 VIS006 :")
!gdalinfo -nomd -mm NETCDF:{chunk1}:/data/vis_06/measured/effective_radiance | grep -E "Min/Max"

print("\n🔴 IR_105 :")
!gdalinfo -nomd -mm NETCDF:{chunk1}:/data/ir_105/measured/effective_radiance | grep -E "Min/Max"

> 💡 *L'option `-mm` affiche les valeurs minimales et maximales trouvées dans le dataset.*

**Interprétation des valeurs** :

- `Computed Min/Max = 303.000, 2844.000` (pour VIS006)
- `Computed Min/Max = -566.000, 3402.000` (pour IR_105)

⚠️ *Les réflectivités supérieures à 10 000 (>100 %) peuvent concerner des nuages glacés épais avec un soleil rasant.*

<div id="2-production-dune-image-tif-canal-vis006"></div>
<div class="alert alert-info alert-success">
<h3 id="2"> 2️⃣ - 🖼️ Production d'une image TIF à partir du canal VIS006</h3>
</div>

In [ ]:
# 📁 Configuration des paramètres de sortie
!mkdir -p RESULTS
output = 'RESULTS'
nom_fic = 'VIS006_mtg_20260614_1200'
redim = '2000'  # Redimensionnement pour l'affichage

print(f"📁 Dossier de sortie : {output}")
print(f"📏 Redimensionnement : {redim} pixels")

<div id="21-etirement-lineaire"></div>

### 🎨 **Étirement linéaire (contrast stretching)**

Nous allons transformer les valeurs physiques (16 bits) en nuances de gris (256 valeurs - 8 bits) avec `gdal_translate` et l'option `-scale` :

```bash
-scale entrée_min entrée_max sortie_min sortie_max
```

Le capteur FCI de MTG numérise les signaux sur 12 bits de dynamique. La valeur maximale de comptage numérique (DN) sera donc 4095 (2^12).

> 📝 *Pour des raisons de compatibilité, la donnée est stockée sur 16 bits.*

In [ ]:
# 🖼️ Génération de l'image VIS006 avec étirement complet
print("🖼️ Génération de l'image VIS006...")

# Étape 1 : Conversion en JPEG avec étirement 0-4095 -> 0-255
!gdal_translate -scale 0 4095 0 255 -ot byte NETCDF:{chunk1}:/data/vis_06/measured/effective_radiance $output/{nom_fic}.jpg 2>/dev/null

# Étape 2 : Redimensionnement pour l'affichage
!convert -resize {redim} $output/{nom_fic}.jpg $output/{nom_fic}_{redim}.jpg 2>/dev/null

# Étape 3 : Affichage
display(IPImage(filename=f"{output}/{nom_fic}_{redim}.jpg", width=1500))

print(f"✅ Image sauvegardée : {output}/{nom_fic}_{redim}.jpg")

### 🔄 Visualisation de plusieurs chunks

Pour visualiser plusieurs chunks (par exemple les chunks 30 à 39), nous faisons une boucle sur chaque fichier.

> 💡 *Chaque chunk représente une bande de 300 lignes. L'assemblage de tous les chunks reconstitue le disque complet.*

In [ ]:
%%bash

#Rappel du répertoire source
#❗ Attention : ce chemin doit-être modifié pour toute arborescence différente
RepSource="../stockage/DATA/20260614/"

# 🧹 Nettoyage du dossier temporaire
rm -f RESULTS/tmp/*

echo "🔄 Extraction des chunks 30 à 39..."

for fic in ${RepSource}/*_003[0-9].nc; do
    #echo "   - Extraction de $(basename ${fic})"
    # Étape 1 : Conversion et étirement
    gdal_translate -q -scale 0 4095 0 255 -ot Byte "NETCDF:\"${fic}\":/data/vis_06/measured/effective_radiance" RESULTS/tmp/$(basename ${fic} .nc).tif 2>/dev/null || true
    # Étape 2 : Redimensionnement
    convert -resize 1200 RESULTS/tmp/$(basename ${fic} .nc).tif RESULTS/$(basename ${fic} .nc).jpg 2>/dev/null
done

echo "✅ Extraction des chunks terminée"

In [ ]:
# 🖼️ Visualisation des chunks extraits
print("🖼️ Visualisation des chunks (ordre chronologique inversé) :")
fichiers = sorted(glob.glob("RESULTS/W_XX*.jpg"), reverse=True)
for i, fichier in enumerate(fichiers[:10]):  # Limite à 10 pour l'affichage
    #print(f"   - Chunk {i+1}: {os.path.basename(fichier)}")
    display(IPImage(filename=fichier, width=1500))

### 🎚️ **Ajustement de la dynamique (windowing)**

En modifiant les bornes d'entrée (`valeur_min_vis` et `valeur_max_vis`), on peut rendre l'image plus lumineuse ou faire ressortir des détails spécifiques.

> 💡 *Cette technique est appelée **windowing** en imagerie médicale et télédétection.*

### 📊 Trouver les valeurs min et max du canal

In [ ]:
# 📊 Extraction des min/max pour un étirement optimal
print("📊 Valeurs min/max du canal VIS006 :")
!gdalinfo -nomd -mm NETCDF:{chunk1}:/data/vis_06/measured/effective_radiance | grep -E "Min/Max"

In [ ]:
%%bash
echo ${RepSource}

In [ ]:
%%bash

#Rappel du répertoire source
#❗ Attention : ce chemin doit-être modifié pour toute arborescence différente
RepSource="../stockage/DATA/20260614/"

# 🎚️ Extraction avec étirement optimisé (valeurs réelles du canal)
valeur_min_vis=303
valeur_max_vis=2844

rm -f RESULTS/tmp/*  # Nettoyage

echo "🎚️ Extraction avec étirement -scale ${valeur_min_vis} ${valeur_max_vis} 0 255"

for fic in ${RepSource}/*_003[0-9].nc; do
    gdal_translate -q -scale ${valeur_min_vis} ${valeur_max_vis} 0 255 -ot Byte "NETCDF:\"${fic}\":/data/vis_06/measured/effective_radiance" RESULTS/tmp/$(basename ${fic} .nc).tif 2>/dev/null || true
    convert -resize 1200 RESULTS/tmp/$(basename ${fic} .nc).tif RESULTS/$(basename ${fic} .nc).jpg 2>/dev/null
done

echo "✅ Extractions terminées pour étirement optimisé"

In [ ]:
# 🖼️ Visualisation des chunks avec étirement optimisé
print("🖼️ Visualisation des chunks avec étirement optimisé :")
fichiers = sorted(glob.glob("RESULTS/W_XX*.jpg"), reverse=True)
for i, fichier in enumerate(fichiers[:10]):
    #print(f"   - Chunk {i+1}: {os.path.basename(fichier)}")
    display(IPImage(filename=fichier, width=1500))

<div id="22-correction-gamma"></div>

### 📐 **Correction Gamma (éclaircissement non linéaire)**

La correction Gamma permet d'éclaircir les zones sombres sans saturer les zones claires.

> 📝 *Cet ajustement permet de modifier la luminosité d'une image de façon non linéaire.*
> On parle aussi de « facteur de contraste ».

| Gamma | Effet |
|-------|-------|
| < 1 | Assombrit l'image |
| = 1 | Aucun changement |
| > 1 | Éclaircit l'image |

> 💡 *GDAL utilise l'option `-exponent` avec comme valeur `1/gamma`.*

**Formule mathématique :**

$$\text{Output} = \text{Input}^{1/\gamma}$$

### 🖼️ Visualisation du plein disque VIS006 (brut)

In [ ]:
# 🖼️ Génération de l'image plein disque VIS006
print("🖼️ Génération de l'image plein disque VIS006...")

!gdal_translate -scale 0 12000 0 255 -ot byte -outsize 2000 2000 {RepSource}/20260614_1200_vis006_16bits.tif {output}/20260614_1200_vis006_2000x2000.tif 2>/dev/null
!convert  {output}/20260614_1200_vis006_2000x2000.tif {output}/20260614_1200_vis006_2000x2000.jpg 2>/dev/null

display(IPImage(filename=f"{output}/20260614_1200_vis006_2000x2000.jpg", width=700))
print("✅ Image plein disque générée")

In [ ]:
# 🎨 Application de la correction gamma (gamma > 1 = éclaircissement)
Gamma = 1.5
ValGamma = 1 / Gamma

print(f"🎨 Application de la correction gamma : γ = {Gamma}")

!gdal_translate -scale -exponent {ValGamma} -ot byte RESULTS/20260614_1200_vis006_2000x2000.tif RESULTS/20260614_1200_vis006_2000x2000_gamma_{Gamma}.tif 2>/dev/null
!convert RESULTS/20260614_1200_vis006_2000x2000_gamma_{Gamma}.tif RESULTS/20260614_1200_vis006_2000x2000_gamma_{Gamma}.jpg 2>/dev/null

display(IPImage(filename=f"{output}/20260614_1200_vis006_2000x2000_gamma_{Gamma}.jpg", width=700))
print(f"✅ Image avec gamma {Gamma} sauvegardée")

In [ ]:
# 🎨 Test avec un gamma plus fort
Gamma = 2.0
ValGamma = 1 / Gamma

print(f"🎨 Application de la correction gamma : γ = {Gamma}")

!gdal_translate -scale 0 10000 0 255 -exponent {ValGamma} -ot byte -outsize 2000 2000 {RepSource}/20260614_1200_vis006_16bits.tif {output}/20260614_1200_vis006_2000x2000_{Gamma}.jpg 2>/dev/null

display(IPImage(filename=f"{output}/20260614_1200_vis006_2000x2000_{Gamma}.jpg", width=700))
print(f"✅ Image avec gamma {Gamma} sauvegardée")

### <div style='background-color: #872459; color: white; padding: 10px; border-radius: 12px; text-align: left;'> ✏️ Exercice 1</div>

### 🔧 **Exercice : Jouer avec la dynamique et le gamma**

Éclaircir et assombrir l'image en jouant sur :
1. **La dynamique** (valeurs min/max de l'étirement) : faire une image très sombre, et une image très claire.
2. **La valeur gamma** (éclaircissement ou assombrissement)

> 💡 *Essayez différentes combinaisons pour observer l'effet sur l'image !*

In [ ]:
# 🔧 À vous de jouer ! Modifiez les paramètres suivants :
Gamma = 1.8  # Essayez 0.5, 1.0, 1.5, 2.0, 3.0
ValGamma = 1 / Gamma

# Vous pouvez aussi modifier les bornes de l'étirement
vmin = 0
vmax = 10000

print(f"🔧 Génération avec gamma = {Gamma} et étirement [{vmin}, {vmax}]")

!gdal_translate -scale {vmin} {vmax} 0 255 -exponent {ValGamma} -ot byte -outsize 2000 2000 {RepSource}/20260614_1200_vis006_16bits.tif {output}/20260614_1200_vis006_2000x2000_gamma_{Gamma}_custom.jpg 2>/dev/null

display(IPImage(filename=f"{output}/20260614_1200_vis006_2000x2000_gamma_{Gamma}_custom.jpg", width=700))
print(f"✅ Image personnalisée sauvegardée")

> 💡 *Le produit opérationnel bénéficie de corrections supplémentaires : diffusion, angle solaire et parallaxe.*

<div id="3-production-dune-image-tif-canal-ir_105"></div>
<div class="alert alert-info alert-success">
<h3 id="3"> 3️⃣ - 🌡️ Production d'une image TIF à partir du canal IR_105</h3>
</div>

### 📊 **Spécificités du canal IR_105**




In [ ]:
# 🔍 Vérification de la présence du canal IR_105
print("🔍 Recherche du canal IR_105 :")
!gdalinfo {chunk1} | grep ir_105

In [ ]:
# 🖼️ Génération de l'image IR_105
nom_fic_ir = 'IR_105_mtg_20260614_1200'

print("🖼️ Génération de l'image IR_105...")

!gdal_translate -scale 0 4095 0 255 -ot byte NETCDF:{chunk1}:/data/ir_105/measured/effective_radiance $output/{nom_fic_ir}.jpg 2>/dev/null
!convert -resize {redim} $output/{nom_fic_ir}.jpg $output/{nom_fic_ir}_{redim}.jpg 2>/dev/null

display(IPImage(filename=f"{output}/{nom_fic_ir}_{redim}.jpg", width=1500))
print(f"✅ Image IR_105 sauvegardée : {output}/{nom_fic_ir}_{redim}.jpg")

In [ ]:
# 📊 Statistiques du canal IR_105
print("📊 Statistiques du canal IR_105 :")
!gdalinfo -nomd -mm NETCDF:{chunk1}:/data/ir_105/measured/effective_radiance | grep Min/Max

In [ ]:
%%bash

#Rappel du répertoire source
#❗ Attention : ce chemin doit-être modifié pour toute arborescence différente
RepSource="../stockage/DATA/20260614/"

# 🌡️ Extraction des chunks IR_105 avec étirement optimisé
valeur_min_ir=566
valeur_max_ir=3402

rm -f RESULTS/tmp/*  # Nettoyage

echo "🌡️ Extraction des chunks IR_105..."

for fic in ${RepSource}/*_003[0-9].nc; do
    gdal_translate -q -scale ${valeur_min_ir} ${valeur_max_ir} 0 255 -ot Byte "NETCDF:\"${fic}\":/data/ir_105/measured/effective_radiance" RESULTS/tmp/$(basename ${fic} .nc).tif 2>/dev/null || true
    convert -resize 1200 RESULTS/tmp/$(basename ${fic} .nc).tif RESULTS/IR_$(basename ${fic} .nc).jpg 2>/dev/null
done

echo "✅ Extraction des chunks IR_105 terminée"

In [ ]:
# 🖼️ Visualisation des chunks IR_105
print("🖼️ Visualisation des chunks IR_105 :")
fichiers = sorted(glob.glob("RESULTS/IR_W_XX*.jpg"), reverse=True)
for i, fichier in enumerate(fichiers[:10]):
   # print(f"   - Chunk {i+1}: {os.path.basename(fichier)}")
    display(IPImage(filename=fichier, width=1500))

### ❓ **Question : cette image vous paraît-elle étrange ?**


### <div style='background-color: #872459; color: white; padding: 10px; border-radius: 12px; text-align: left;'> ✏️ Exercice 2</div>

### 🎨 **Exercice : Produire une image IR_105 avec une autre dynamique de couleur**


### 🌍 **Convention de représentation des températures**

| Région | Convention | Dynamique (`-scale`) |
|--------|------------|----------------------|
| **Europe/Afrique** | Nuages froids = blancs | `min max 255 0` (inversé) |
| **USA** | Nuages chauds = blancs | `min max 0 255` (standard) |

> 💡 *Pour la convention européenne/africaine, il suffit d'inverser les bornes de sortie : `-scale min max 255 0`*

---

## ✅ **Résumé des bonnes pratiques**

| Action | Commande / Option |
|--------|-------------------|
| Visualiser un NetCDF | `gdalinfo fichier.nc` |
| Extraire un canal | `NETCDF:\"fichier.nc\":canal` |
| Appliquer un étirement | `-scale min_in max_in min_out max_out` |
| Appliquer un gamma | `-exponent 1/gamma` |
| Inverser l'échelle | `-scale min_in max_in 255 0` |
| Extraire une valeur ponctuelle | `gdallocationinfo -wgs84 ... lon lat` |
| Interpolation cubique | `-r cubic` |

---

## 🔗 **Ressources complémentaires**

| Lien | Description |
|------|-------------|
| [Documentation GDAL](https://gdal.org) | Référence complète |
| [Galerie CMS](http://e-sat.cms.meteo.fr/GALERIE/index.html) | Galerie CMS |
| [Comparateur d'images](http://e-sat.cms.meteo.fr/GALERIE/slider4_vierge.html) | Outil de comparaison |

---

✅ **Fin du TP**

---

## 📝 **Synthèse des acquis**

| Compétence | Niveau acquis |
|------------|---------------|
| 🔎 Lecture et analyse de fichiers NetCDF | ✅ |
| 🖼️ Production d'images à partir de canaux | ✅ |
| 🎚️ Ajustement de la dynamique (étirement) | ✅ |
| 🌈 Correction gamma | ✅ |
| 🌡️ Convention de représentation IR | ✅ |
